# This notebook demonstrates the problem described in Issue [#578](https://github.com/ciemss/pyciemss/issues/578)

### Load dependencies

In [1]:
import os
import json
import sympy
import torch
import numpy as np
import pandas as pd
from typing import Dict, List, Callable

# MIRA modeling dependencies
from mira.metamodel import *
from mira.metamodel.ops import stratify
from mira.examples.concepts import susceptible, exposed, infected, recovered
from mira.modeling import Model
from mira.modeling.amr.petrinet import AMRPetriNetModel, template_model_to_petrinet_json
from mira.sources.amr.petrinet import template_model_from_amr_json
from mira.metamodel.io import model_to_json_file, model_from_json_file

# PyCIEMSS dependencies
import pyciemss
import pyciemss.visuals.plots as plots
import pyciemss.visuals.vega as vega
import pyciemss.visuals.trajectories as trajectories
from pyciemss.integration_utils.intervention_builder import (
    param_value_objective,
    start_time_objective,
)

### Define a simple SIR model

In [2]:
person_units = lambda: Unit(expression=sympy.Symbol('person'))
day_units = lambda: Unit(expression=sympy.Symbol('day'))
per_day_units = lambda: Unit(expression=1/sympy.Symbol('day'))
dimensionless_units = lambda: Unit(expression=sympy.Integer('1'))

c = {
    # human population
    "S": Concept(name="S", units=person_units(), identifiers={"ido": "0000514"}),
    "I": Concept(name="I", units=person_units(), identifiers={"ido": "0000511"}),
    "R": Concept(name="R", units=person_units(), identifiers={"ido": "0000592"}),
}

for concept in c:
    c[concept].name = concept

parameters = {
    # transition rates for human population
    'beta': Parameter(name='beta', value=sympy.Float(0.0025), units=per_day_units(),
                      distribution=Distribution(type='StandardUniform1',
                                                parameters={'minimum': 0.002,
                                                            'maximum': 0.003})),  # Transmission rate S -> I
    
    'gamma': Parameter(name='gamma', value=sympy.Float(0.07), units=per_day_units()),  # Rate of progressing I -> R
}

S, I, R, beta, gamma = sympy.symbols('S I R beta gamma')

initials = {
    "S": Initial(concept=c["S"], expression=999.0),
    "I": Initial(concept=c["I"], expression=1.0),
    "R": Initial(concept=c["R"], expression=0.0),
}

# Transitions
##### S -> I
si = ControlledConversion(
    subject=c['S'],
    outcome=c['I'],
    controller=c['I'],
    rate_law=beta*S*I
)


#### I -> R
ir = NaturalConversion(
    subject=c['I'],
    outcome=c['R'],
    rate_law=gamma*I
)

### Define model observables: `observables1` uses the of `beta` "value", `observables2` uses the parameter (ostensibly the value drawn from the distribution provided)

In [3]:
observables1 = {
    'incident_cases': Observable(name='incident_cases', expression=parameters["beta"].value*S*I),
    'beta_param': Observable(name='beta_param', expression=parameters["beta"].value)
}

observables2 = {
    'incident_cases': Observable(name='incident_cases', expression=beta*S*I),
    'beta_param': Observable(name='beta_param', expression=beta)
}

### Save model petrinet AMRs

In [4]:
sir1 = TemplateModel(
    templates=[
        si,
        ir
    ],
    parameters=parameters,
    initials=initials,
    time=Time(name='t', units=day_units()),
    observables=observables1,
    annotations=Annotations(name='SIR model with observables1')
)

# Save as JSON
with open("SIR_v1.json", 'w') as fh:
    json.dump(template_model_to_petrinet_json(sir1), fh, indent=1)


sir2 = TemplateModel(
    templates=[
        si,
        ir
    ],
    parameters=parameters,
    initials=initials,
    time=Time(name='t', units=day_units()),
    observables=observables2,
    annotations=Annotations(name='SIR model with observables2')
)

# Save as JSON
with open("SIR_v2.json", 'w') as fh:
    json.dump(template_model_to_petrinet_json(sir2), fh, indent=1)

### Sample from model 1 (defined with `observables1`)

In [5]:
model1 = "SIR_v1.json"
num_samples = 100
start_time = 0.0
end_time = 50.0
logging_step_size = 1.0

result1 = pyciemss.sample(model1, end_time, logging_step_size, num_samples, start_time=start_time)
display(result1['data'].head())

# # Plot results for human states
# schema = plots.trajectories(result1["data"], keep=".*_state")
# plots.save_schema(schema, "_schema.json")
# plots.ipy_display(schema, dpi=150)

,timepoint_id,sample_id,timepoint_unknown,persistent_beta_param,S_state,I_state,R_state,incident_cases_observable_state,beta_param_observable_state
0,0,0,0.0,0.002339,999.000000,1.000000,0.000000,2.497500,0.0025
1,1,0,1.0,0.002339,990.153931,9.579865,0.266212,23.713852,0.0025
2,2,0,2.0,0.002339,912.175049,85.103645,2.721358,194.073547,0.0025
3,3,0,3.0,0.002339,520.608337,459.884125,19.507311,598.548767,0.0025
4,4,0,4.0,0.002339,106.538040,826.470276,66.991768,220.126312,0.0025


### Sample from model 2 (defined with `observables2`)

In [6]:
model2 = "SIR_v2.json"
num_samples = 100
start_time = 0.0
end_time = 50.0
logging_step_size = 1.0

result2 = pyciemss.sample(model2, end_time, logging_step_size, num_samples, start_time=start_time)
display(result2['data'].head())

# # Plot results for human states
# schema = plots.trajectories(result2["data"], keep=".*_state")
# plots.save_schema(schema, "_schema.json")
# plots.ipy_display(schema, dpi=150)

ERROR:root:
                ###############################

                There was an exception in pyciemss

                Error occured in function: sample

                Function docs : 
    Load a model from a file, compile it into a probabilistic program, and sample from it.

    Args:
        model_path_or_json: Union[str, Dict]
            - A path to a AMR model file or JSON containing a model in AMR form.
        end_time: float
            - The end time of the sampled simulation.
        logging_step_size: float
            - The step size to use for logging the trajectory.
        num_samples: int
            - The number of samples to draw from the model.
        solver_method: str
            - The method to use for solving the ODE. See torchdiffeq's `odeint` method for more details.
            - If performance is incredibly slow, we suggest using `euler` to debug.
              If using `euler` results in faster simulation, the issue is likely that the model is s

ValueError: The model observables could not be evaluated. Please check the model definition. This could be due to to a missing state variable or parameter, or an error in the observables definition.
                               Trace Shapes:  
                                Param Sites:  
numeric_initial_state_func$$$_nodes.0._value  
numeric_initial_state_func$$$_nodes.1._value  
numeric_initial_state_func$$$_nodes.2._value  
                               Sample Sites:  
                        persistent_beta dist |
                                       value |